# Column A Twin Walkthrough

Guided tour of the Skogestad Column A dynamic twin shipped in Phase 1. This notebook covers:

1. Parameters and steady-state initialization
2. Per-stage composition profile at the published SS
3. Open-loop step response (matches the Octave reference)
4. LV closed-loop response to a feed-flow disturbance
5. Operating-window sweep summary

All numerical anchors trace to Skogestad & Morari 1988 and Skogestad 1997; see `src/industrial_ai/twin/column_a/assumptions.md` for the full list of modeling assumptions.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from industrial_ai.twin.column_a import (
    DEFAULT_PARAMETERS,
    compute_steady_state_by_integration,
    integrate_open_loop,
)
from industrial_ai.twin.column_a.configurations.lv import assemble_inputs_lv
from industrial_ai.twin.column_a.steady_state import (
    flat_initial_state,
    nominal_inputs,
)
from industrial_ai.twin.simulate import ScenarioStep, simulate_lv_closed_loop

p = DEFAULT_PARAMETERS
print(
    f"Column A: {p.NT} stages, feed at stage {p.NF}, alpha = {p.alpha}, "
    f"L0 = {p.nominal_reflux_L0_kmol_per_min:.4f}, V0 = {p.nominal_boilup_V0_kmol_per_min:.4f}"
)


## 1. Steady state from a flat profile

The Skogestad cold-start procedure runs the column from a flat `x = 0.5` profile under the nominal inputs for 20 000 minutes. The result is the published steady state with 99 % top and bottom purities.

In [ ]:
X0 = flat_initial_state(p)
ss = compute_steady_state_by_integration(parameters=p, inputs=nominal_inputs(p))
print(f"converged: {ss.success} | residual = {ss.residual_norm:.2e}")
print(
    f"y_D = {ss.X[p.NT - 1]:.5f} | x_B = {ss.X[0]:.5f} | "
    f"MD = {ss.X[2 * p.NT - 1]:.4f} | MB = {ss.X[p.NT]:.4f}"
)


## 2. Per-stage composition profile

The S-shape is the classical signature of a binary distillation column at high purity.

In [ ]:
stages = np.arange(p.NT) + 1  # 1-indexed in Skogestad's convention
compositions = ss.X[: p.NT]

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(compositions, stages, marker="o")
ax.axhline(p.NF, color="grey", linestyle="--", alpha=0.5, label=f"Feed (stage {p.NF})")
ax.set_xlabel("Light-component mole fraction x")
ax.set_ylabel("Stage (1 = reboiler, NT = condenser)")
ax.set_title("Column A steady-state composition profile")
ax.legend()
plt.tight_layout()


## 3. Open-loop step response (L +1 %)

Reproduces the Octave reference trajectory in `tests/test_column_a_against_matlab.py`. With +1 % reflux the top composition rises and the bottom composition rises slightly (more light component is held inside the column).

In [ ]:
u_nom = nominal_inputs(p)
u_step = u_nom.copy()
u_step[0] *= 1.01  # +1 % on LT


def step_inputs(t, X):
    return u_step


t_eval = np.linspace(0, 300, 301)
result = integrate_open_loop(
    X0=ss.X, t_span=(0.0, 300.0), inputs_fn=step_inputs, t_eval=t_eval
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(result.t, result.X[:, p.NT - 1], label="y_D (top)")
ax.plot(result.t, result.X[:, 0], label="x_B (bottom)")
ax.set_xlabel("Time (min)")
ax.set_ylabel("Composition (mole fraction)")
ax.set_title("Open-loop response to +1 % step in LT")
ax.legend()
plt.tight_layout()


## 4. LV closed-loop response to a feed-flow disturbance

Now bring the regulatory PIDs online and exercise the closed loop with a +20 % step in feed flow at t = 5 min.

In [ ]:
def feed_step_scenario(t):
    F = p.nominal_feed_F_kmol_per_min * (1.2 if t >= 5.0 else 1.0)
    return ScenarioStep(y_D_setpoint=0.99, x_B_setpoint=0.01, F=F, zF=0.5, qF=1.0)


sim = simulate_lv_closed_loop(
    X0=ss.X,
    scenario=feed_step_scenario,
    duration_min=60.0,
    tick_dt_min=0.05,
)
print(f"sim succeeded: {sim.success} | max wall-clock per tick: "
      f"{sim.cycle_wall_clock_seconds.max() * 1e3:.2f} ms")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax1.plot(sim.t, sim.y_D, label="y_D")
ax1.plot(sim.t, sim.x_B, label="x_B")
ax1.axvline(5.0, color="grey", linestyle="--", alpha=0.5)
ax1.set_ylabel("Composition")
ax1.legend()
ax2.plot(sim.t[1:], sim.inputs[:, 0], label="LT (reflux)")
ax2.plot(sim.t[1:], sim.inputs[:, 1], label="VB (boilup)")
ax2.axvline(5.0, color="grey", linestyle="--", alpha=0.5)
ax2.set_xlabel("Time (min)")
ax2.set_ylabel("Flow (kmol/min)")
ax2.legend()
plt.tight_layout()


## 5. Operating-window sweep summary

1080 grid points warm-started from the published SS, each re-converged via Newton-Krylov with an integration fallback for ill-conditioned (LT, VB) combinations. Generated by `tools/run_operating_window_sweep.py`.

In [ ]:
csv_path = Path("..") / "data" / "baseline_operating_window.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"{len(df)} grid points; convergence = {df['success'].mean():.1%}")
    print(
        df[["F", "zF", "LT", "VB", "y_D", "x_B", "residual_norm"]]
        .describe()
        .round(4)
    )
else:
    print(
        "Run `uv run python tools/run_operating_window_sweep.py` first to generate the CSV."
    )


## Next steps

Phase 2 plugs a linear-MPC supervisor in above the same regulatory PID layer (`src/industrial_ai/control/c1_linear_mpc.py`, derived from `column_a/linearize.py`). The Phase 1 twin remains the substrate — every Phase 2/3/4/5 configuration calls into the same `simulate_lv_closed_loop` plus the data-logging contract demonstrated above.